<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D0%9E%D0%B1%D1%83%D1%87%D0%B0%D1%8E%D1%89%D0%B8%D0%B9_%D0%BD%D0%B0%D0%B1%D0%BE%D1%80__%D0%A7%D0%B0%D1%81%D1%82%D1%8C_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Цель работы:**  
Разработать модель, способную предсказывать состав следующего заказа пользователя на основе анализа его истории покупок. Это позволит повысить персонализацию сервиса и улучшить пользовательский опыт, а также оптимизировать процессы формирования корзины и планирования закупок.


# Введение и постановка задачи

Современные сервисы доставки продуктов активно используют данные о покупках пользователей для улучшения качества и персонализации сервиса. Анализ истории заказов позволяет прогнозировать, какие категории товаров пользователь может захотеть приобрести в следующий раз.

Целью данного проекта является создание модели, способной по истории заказов прогнозировать состав будущего заказа пользователя. Такая рекомендация поможет клиенту сэкономить время на формировании корзины, избежать забытых позиций и повысить удобство планирования закупок.

Задача формализована как многоклассовая классификация: необходимо предсказать для каждой пары (пользователь, категория), будет ли категория включена в следующий заказ. Для оценки качества модели используется метрика F1-score, которая учитывает баланс между точностью и полнотой предсказаний.

Данные и постановка задачи основаны на открытом соревновании в области электронных продаж.


# Описание набора данных

В проекте используется история заказов 20 000 пользователей, разделённая на тренировочную и тестовую выборки по дате. Тестовая выборка содержит заказы после определённой даты отсечки.

Основной тренировочный файл содержит следующие данные:  
- **user_id** — уникальный идентификатор пользователя  
- **order_completed_at** — дата и время завершения заказа  
- **cart** — категория товара, входящего в заказ (уникальные категории)

Задача — для каждой пары (пользователь, категория), встречающейся в тестовой выборке, предсказать бинарный признак: будет ли категория присутствовать в следующем заказе пользователя.

Идентификаторы пар представлены в формате `"{user_id};{category_id}"`, что учитывается при обработке данных.

Данные подготовлены на основе истории заказов с учётом особенностей временного разделения выборок.


In [278]:
# Подключение Google Drive для доступа к данным и сохранения результатов
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [279]:
# Создание структуры проекта
import os
from pathlib import Path
import pandas as pd
import numpy as np


# Основная папка проекта (как вы указали)
project_path = "/content/drive/MyDrive/Colab Notebooks/sm"

# Чек-лист папок и файлов
structure = [
    "data/raw",          # Исходные данные
    "data/processed",    # Очищенные данные
    "notebooks",         # Jupyter-тетради
    "src/models",        # Код моделей
    "src/utils",         # Утилиты
    "scripts",           # Скрипты обработки
]

# Создаём все директории
for folder in structure:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)


In [280]:
# Функция для рекурсивного вывода структуры папок проекта
def print_tree(root, prefix=""):
    files = sorted(os.listdir(root))
    for i, name in enumerate(files):
        path = os.path.join(root, name)
        is_last = (i == len(files) - 1)
        branch = "└── " if is_last else "├── "
        print(prefix + branch + name + ("/" if os.path.isdir(path) else ""))
        if os.path.isdir(path):
            new_prefix = prefix + ("    " if is_last else "│   ")
            print_tree(path, new_prefix)

print("\n=== Структура проекта (project_path) ===")
print_tree(project_path)


=== Структура проекта (project_path) ===
├── data/
│   ├── processed/
│   │   └── full_orders.parquet
│   └── raw/
│       ├── sample_submission.csv
│       └── train.csv
├── notebooks/
├── plan/
├── scripts/
└── src/
    ├── models/
    └── utils/


In [281]:
# Блок: Загрузка и первичный анализ train.csv
import pandas as pd
import os


train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/sm/data/raw/train.csv")

print("Структура и информация о train.csv:")
print(train.info())

Структура и информация о train.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3123064 entries, 0 to 3123063
Data columns (total 3 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   user_id             int64 
 1   order_completed_at  object
 2   cart                int64 
dtypes: int64(2), object(1)
memory usage: 71.5+ MB
None


In [282]:
train

,user_id,order_completed_at,cart
0,2,2015-03-22 09:25:46,399
1,2,2015-03-22 09:25:46,14
2,2,2015-03-22 09:25:46,198
3,2,2015-03-22 09:25:46,88
4,2,2015-03-22 09:25:46,157
...,...,...,...
3123059,12702,2020-09-03 23:45:45,441
3123060,12702,2020-09-03 23:45:45,92
3123061,12702,2020-09-03 23:45:45,431
3123062,12702,2020-09-03 23:45:45,24


In [283]:
print(f"Уникальных пользователей: {train['user_id'].nunique()}")

Уникальных пользователей: 20000


In [284]:
# Блок: Дополнительная статистика по train.csv

print("\nДополнительная статистика по train.csv:")

orders_per_user = train['user_id'].value_counts()
print("\nРаспределение количества заказов на пользователя:")
print(orders_per_user.describe())

print(f"\nУникальных категорий (корзин): {train['cart'].nunique()}")

train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])
print("\nСтатистика по датам заказов:")
print(f"Период с {train['order_completed_at'].min()} по {train['order_completed_at'].max()}")



Дополнительная статистика по train.csv:

Распределение количества заказов на пользователя:
count    20000.000000
mean       156.153200
std        200.840781
min          3.000000
25%         48.000000
50%         88.000000
75%        181.000000
max       3508.000000
Name: count, dtype: float64

Уникальных категорий (корзин): 881

Статистика по датам заказов:
Период с 2015-03-22 09:25:46 по 2020-09-03 23:45:45


In [285]:
# Преобразуем строку в формат даты
train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])

# Обрезаем дату, убирая время
train['order_completed_at'] = train['order_completed_at'].dt.floor('D')

# Проверим, как выглядит дата после обрезки
print(train['order_completed_at'].head())

0   2015-03-22
1   2015-03-22
2   2015-03-22
3   2015-03-22
4   2015-03-22
Name: order_completed_at, dtype: datetime64[ns]


In [286]:
# Установим стартовую дату
last_date = pd.to_datetime('2020-09-03')
interval_length_days = 30

# Проверим периоды, двигаясь назад
current_date = last_date
is_stable = False

while current_date >= train['order_completed_at'].min():
    # Рассчитаем предыдущие 30 дней
    prev_date = current_date - pd.Timedelta(interval_length_days, unit='D')

    # Отфильтруем заказы в рассматриваемом диапазоне
    period_orders = train[
        (train['order_completed_at'] >= prev_date) &
        (train['order_completed_at'] <= current_date)
    ]

    # Сформируем дни в этом диапазоне
    days_in_range = pd.date_range(start=prev_date, end=current_date)

    # Проверим наличие заказов в КАЖДЫЙ день периода
    missing_days = set(days_in_range) - set(period_orders['order_completed_at'].dt.normalize())

    # Количество заказов в периоде
    total_orders = len(period_orders)

    # Выводим результат
    if len(missing_days) == 0:
        print(f"Период с {prev_date} по {current_date} стабилен (каждый день есть заказы) - {total_orders} заказов")
    else:
        print(f"Период с {prev_date} по {current_date} нестабилен ({len(missing_days)} пропусков) - {total_orders} заказов")
        break

    # Передвинем дату назад
    current_date = prev_date

Период с 2020-08-04 00:00:00 по 2020-09-03 00:00:00 стабилен (каждый день есть заказы) - 464685 заказов
Период с 2020-07-05 00:00:00 по 2020-08-04 00:00:00 стабилен (каждый день есть заказы) - 472867 заказов
Период с 2020-06-05 00:00:00 по 2020-07-05 00:00:00 стабилен (каждый день есть заказы) - 454896 заказов
Период с 2020-05-06 00:00:00 по 2020-06-05 00:00:00 стабилен (каждый день есть заказы) - 357086 заказов
Период с 2020-04-06 00:00:00 по 2020-05-06 00:00:00 стабилен (каждый день есть заказы) - 293168 заказов
Период с 2020-03-07 00:00:00 по 2020-04-06 00:00:00 стабилен (каждый день есть заказы) - 200635 заказов
Период с 2020-02-06 00:00:00 по 2020-03-07 00:00:00 стабилен (каждый день есть заказы) - 154535 заказов
Период с 2020-01-07 00:00:00 по 2020-02-06 00:00:00 стабилен (каждый день есть заказы) - 135627 заказов
Период с 2019-12-08 00:00:00 по 2020-01-07 00:00:00 стабилен (каждый день есть заказы) - 138539 заказов
Период с 2019-11-08 00:00:00 по 2019-12-08 00:00:00 стабилен (ка

In [287]:
# Установим границы периода
start_date = pd.to_datetime('2019-09-09')
end_date = pd.to_datetime('2020-09-03')

# Фильтруем только те заказы, которые попадают в указанный период
train = train[
    (train['order_completed_at'] >= start_date) &
    (train['order_completed_at'] <= end_date)
]

In [288]:
train

,user_id,order_completed_at,cart
167611,2522,2019-09-09,798
167612,2522,2019-09-09,92
167613,2522,2019-09-09,19
167614,2522,2019-09-09,382
167615,2522,2019-09-09,22
...,...,...,...
3123059,12702,2020-09-03,441
3123060,12702,2020-09-03,92
3123061,12702,2020-09-03,431
3123062,12702,2020-09-03,24


In [289]:
# Установка границ периода
start_date = pd.to_datetime('2019-08-10')
end_date = pd.to_datetime('2020-09-03')

# Фильтрация данных
train = train[
    (train['order_completed_at'] >= start_date) &
    (train['order_completed_at'] <= end_date)
]

# Подсчёт статистики по пользователям
users_stats = train['user_id'].value_counts()

# Количестве уникальных пользователей
unique_users = train['user_id'].nunique()

# Количество уникальных корзин
unique_carts = train['cart'].nunique()

# Статистика по датам
start_order_date = train['order_completed_at'].min()
end_order_date = train['order_completed_at'].max()

# Выводим статистику
print("Дополнительная статистика:")
print("Распределение количества заказов на пользователя:")
print(users_stats.describe())

# Теперь правильное количество уникальных пользователей
print("\nУникальных пользователей:", unique_users)

# Количество уникальных корзин
print("\nУникальных категорий (корзин):", unique_carts)

# Статистика по датам
print("\nСтатистика по датам заказов:")
print(f"Период с {start_order_date} по {end_order_date}")

Дополнительная статистика:
Распределение количества заказов на пользователя:
count    20000.000000
mean       147.772650
std        180.092583
min          1.000000
25%         47.000000
50%         85.000000
75%        175.000000
max       2566.000000
Name: count, dtype: float64

Уникальных пользователей: 20000

Уникальных категорий (корзин): 870

Статистика по датам заказов:
Период с 2019-09-09 00:00:00 по 2020-09-03 00:00:00


In [290]:
# Блок: Загрузка и первичный анализ sample_submission.csv
sub = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/sm/data/raw/sample_submission.csv")

print("Структура и информация:")
print(sub.info())

Структура и информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 790449 entries, 0 to 790448
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   id      790449 non-null  object
 1   target  790449 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 12.1+ MB
None


In [291]:
sub

,id,target
0,0;133,0
1,0;5,1
2,0;10,0
3,0;396,1
4,0;14,0
...,...,...
790444,19998;26,0
790445,19998;31,0
790446,19998;29,1
790447,19998;798,1


In [292]:
# Границы дат без времени (чистые даты)
last_date = train['order_completed_at'].max().floor('D')
first_date = train['order_completed_at'].min().floor('D')

# Подсчет разницы в днях
diff_in_days = (last_date - first_date).days

# Подсчет полного количества интервалов (целых частей по 30 дней)
num_full_intervals = diff_in_days // 30

# Остаток дней после целого деления
remaining_days = diff_in_days % 30

# Распечатываем результат
print("Максимальная дата:", last_date)
print("Минимальная дата:", first_date)
print("Всего дней между датами:", diff_in_days)
print("Количество полных интервалов по 30 дней:", num_full_intervals)
print("Оставшиеся дни после деления:", remaining_days)

Максимальная дата: 2020-09-03 00:00:00
Минимальная дата: 2019-09-09 00:00:00
Всего дней между датами: 360
Количество полных интервалов по 30 дней: 12
Оставшиеся дни после деления: 0


In [293]:
# Зафиксированная максимальная дата
last_date = full_orders['order_completed_at'].max().floor('D')

# Формируем временные интервалы вручную
current_date = last_date
intervals = []

# Заполняем полный диапазон интервалов
for _ in range(num_full_intervals):
    prev_date = current_date - pd.Timedelta(30, unit='D')
    intervals.append(pd.Interval(prev_date, current_date, closed='right'))
    current_date = prev_date

# Отсортируем интервалы от старых к новым
intervals.reverse()

# Выведем первые 12 интервалов
print("Первые 12 интервалов:")
for interval in intervals[:12]:
    print(interval)

Первые 12 интервалов:
(2019-09-09 00:00:00, 2019-10-09 00:00:00]
(2019-10-09 00:00:00, 2019-11-08 00:00:00]
(2019-11-08 00:00:00, 2019-12-08 00:00:00]
(2019-12-08 00:00:00, 2020-01-07 00:00:00]
(2020-01-07 00:00:00, 2020-02-06 00:00:00]
(2020-02-06 00:00:00, 2020-03-07 00:00:00]
(2020-03-07 00:00:00, 2020-04-06 00:00:00]
(2020-04-06 00:00:00, 2020-05-06 00:00:00]
(2020-05-06 00:00:00, 2020-06-05 00:00:00]
(2020-06-05 00:00:00, 2020-07-05 00:00:00]
(2020-07-05 00:00:00, 2020-08-04 00:00:00]
(2020-08-04 00:00:00, 2020-09-03 00:00:00]


In [294]:
# Назначаем интервалы каждой записи
train['custom_interval'] = pd.cut(train['order_completed_at'], bins=intervals, include_lowest=True, labels=range(len(intervals)), ordered=False)

In [296]:
train

,user_id,order_completed_at,cart,custom_interval
167611,2522,2019-09-09,798,NaN
167612,2522,2019-09-09,92,NaN
167613,2522,2019-09-09,19,NaN
167614,2522,2019-09-09,382,NaN
167615,2522,2019-09-09,22,NaN
...,...,...,...,...
3123059,12702,2020-09-03,441,"(2020-08-04 00:00:00, 2020-09-03 00:00:00]"
3123060,12702,2020-09-03,92,"(2020-08-04 00:00:00, 2020-09-03 00:00:00]"
3123061,12702,2020-09-03,431,"(2020-08-04 00:00:00, 2020-09-03 00:00:00]"
3123062,12702,2020-09-03,24,"(2020-08-04 00:00:00, 2020-09-03 00:00:00]"
